# AI Avatar Platform — Kaggle GPU runner

Clone the repo, install deps, download model weights, verify the GPU env, run a real
end-to-end **create avatar → generate video** pass on the GPU, and (optionally) expose
the whole site via a Cloudflare quick tunnel.

**Settings → Accelerator: GPU T4 x2, Internet: ON.** See `docs/deployment/kaggle.md`.

In [ ]:
# 1. Clone + install
REPO = 'https://github.com/<you>/ai-avatar-platform.git'
!git clone $REPO
%cd ai-avatar-platform/backend
!pip install -e . -q
!apt-get -y install ffmpeg >/dev/null

In [ ]:
# 2. Download weights (idempotent/resumable) + verify environment.
#    If you cached weights to a Kaggle Dataset, mount it and skip the download.
import os
os.environ.setdefault('HF_TOKEN', '')   # set if a model repo is gated
!python ../ml_models/download_models.py
!python ../ml_models/verify_env.py

In [ ]:
# 3. Migrate the DB. Use the real backend (GPU present).
import os
os.environ['PIPELINE_BACKEND'] = 'real'   # 'stub' for a no-GPU dry run
os.environ['JWT_SECRET'] = 'replace-with-a-long-random-value'
!alembic upgrade head

In [ ]:
# 4. End-to-end smoke via the Python API (no HTTP): create avatar from a sample
#    video, run the avatar pipeline, then generate a video and confirm the MP4.
#    Provide a 2–5 min sample clip path in SAMPLE_VIDEO.
from app.db.session import SessionLocal
from app.models.user import User
from app.models.avatar import Avatar
from app.models.enums import AvatarStatus
from app.services import script_service, avatar_service
from app.pipelines.avatar_creation.runner import run_avatar_pipeline
from app.queue import db_queue
from app.pipelines.generation.orchestrator import run_generation_job
from app.models.training_video import TrainingVideo
from app.models.enums import VideoStatus

SAMPLE_VIDEO = '/kaggle/input/<your-sample>/clip.mp4'   # 2–5 min talking-head clip

db = SessionLocal()
user = User(email='kaggle@example.com', hashed_password='x', is_active=True)
db.add(user); db.flush()
avatar = avatar_service.create_avatar(db, user_id=user.id, name='Kaggle Avatar')
script_service.create_script_for_avatar(db, user_id=user.id, avatar_id=avatar.id)
db.add(TrainingVideo(user_id=user.id, avatar_id=avatar.id, file_path=SAMPLE_VIDEO, status=VideoStatus.uploaded))
db.commit(); aid = avatar.id; db.close()

run_avatar_pipeline(aid)                       # face + voice + profile (GPU)
db = SessionLocal(); print('avatar status:', db.get(Avatar, aid).status); db.close()

In [ ]:
# 5. Generate a video for the ready avatar and confirm the output MP4.
from app.core.paths import output_video_final_path
db = SessionLocal()
job = db_queue.enqueue_job(db, user_id=user.id, avatar_id=aid, script_text='Hello from my cloned avatar on a free GPU!')
claimed = db_queue.claim_next_job(db)
video = run_generation_job(db, claimed)
print('job:', claimed.status, 'video:', video.id if video else None)
print('output exists:', output_video_final_path(claimed.id).exists())
db.close()

In [ ]:
# 6. (Optional) Serve the whole site publicly for this session via Cloudflare.
import subprocess
subprocess.Popen(['python', '-m', 'worker.main'])
subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '7860'])
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
# Prints a public https URL — that is your website for this session:
!./cloudflared tunnel --url http://localhost:7860

## 7. Cache weights to a Kaggle Dataset
After `download_models.py`, use **File → Add or upload data → New Dataset** from
`ml_models/weights/`. Next session, **Add Data → your dataset** mounts it read-only
at `/kaggle/input/...`; point `ML_MODELS_DIR` there and skip the download.